# Notebook 04 — Results Analysis and Visualisation
**MRes Computing and Artificial Intelligence — MRES7015**

---
Loads all results from Notebooks 02 and 03 and produces:
- Figure 5.1: Convergence comparison chart
- Figure 5.2: Algorithm performance bar chart
- Figure 5.3: Privacy-utility trade-off (DP results)
- Figure 5.4: HE overhead summary
- Table 5.1: Full results summary CSV

Run after Notebooks 02 and 03.


## Stage 1 — Setup and Load Results

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': 0.6, 'figure.facecolor': 'white',
    'axes.facecolor': 'white', 'grid.color': '#DDDDDD',
    'grid.linewidth': 0.4, 'savefig.dpi': 220,
    'savefig.bbox': 'tight', 'savefig.facecolor': 'white',
})

DRIVE_BASE  = '/content/drive/MyDrive/FL_Dissertation/'
RESULTS_DIR = DRIVE_BASE + 'results/'
OUTPUTS_DIR = DRIVE_BASE + 'outputs/'
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Load all simulation results
all_results = {}
for fname in sorted(os.listdir(RESULTS_DIR)):
    if fname.endswith('.json'):
        with open(RESULTS_DIR + fname) as f:
            all_results[fname.replace('.json','')] = json.load(f)

print(f"Result files loaded: {len(all_results)}")
for k in sorted(all_results.keys()):
    print(f"  {k}")

Mounted at /content/drive
Result files loaded: 16
  dp_results
  fedavg_ext_noniid
  fedavg_iid
  fedavg_mod_noniid
  fedprox_ext_noniid
  fedprox_iid
  fedprox_mod_noniid
  he_results
  mia_results
  perfedavg_ext_noniid
  perfedavg_iid
  perfedavg_mod_noniid
  scaffold_ext_noniid
  scaffold_iid
  scaffold_mod_noniid
  test_evaluation


## Stage 2 — Results Summary Table (Table 5.1)

In [ ]:
alg_labels = {
    'fedavg':    'FedAvg',
    'fedprox':   'FedProx',
    'scaffold':  'SCAFFOLD',
    'perfedavg': 'Per-FedAvg'
}
cond_labels = {
    'iid':         'IID (α=1.0)',
    'mod_noniid':  'Moderate non-IID (α=0.5)',
    'ext_noniid':  'Extreme non-IID (α=0.1)'
}

rows = []
for key, result in all_results.items():
    if not isinstance(result, dict): continue
    alg  = result.get('algorithm','')
    cond = result.get('condition','')
    if alg not in alg_labels: continue

    auc_series  = [v for _,v in result.get('auc',[])]
    loss_series = [v for _,v in result.get('losses',[])]

    rows.append({
        'Algorithm':     alg_labels.get(alg, alg),
        'Condition':     cond_labels.get(cond, cond),
        'Final AUC-ROC': round(auc_series[-1], 4)  if auc_series  else 0.0000,
        'Final Loss':    round(loss_series[-1], 4) if loss_series else 'N/A',
        'Rounds':        len(loss_series),
    })

# Fix: Check if rows is empty before creating the DataFrame to avoid KeyError on sort_values
if not rows:
    df_results = pd.DataFrame(columns=['Algorithm', 'Condition', 'Final AUC-ROC', 'Final Loss', 'Rounds'])
else:
    df_results = pd.DataFrame(rows).sort_values(['Algorithm','Condition'])

print("TABLE 5.1 — FL Algorithm Performance Summary")
print("="*70)
print(df_results.to_string(index=False))

df_results.to_csv(OUTPUTS_DIR + 'Table_5_1_Results_Summary.csv', index=False)
print(f"\nTable saved to: {OUTPUTS_DIR}Table_5_1_Results_Summary.csv")

TABLE 5.1 — FL Algorithm Performance Summary
 Algorithm                Condition  Final AUC-ROC  Final Loss  Rounds
    FedAvg  Extreme non-IID (α=0.1)            0.0      0.1591      50
    FedAvg              IID (α=1.0)            0.0      0.2248      50
    FedAvg Moderate non-IID (α=0.5)            0.0      0.1243      50
   FedProx  Extreme non-IID (α=0.1)            0.0      0.1574      50
   FedProx              IID (α=1.0)            0.0      0.1617      50
   FedProx Moderate non-IID (α=0.5)            0.0      0.1072      50
Per-FedAvg  Extreme non-IID (α=0.1)            0.0      0.1605      50
Per-FedAvg              IID (α=1.0)            0.0      0.2327      50
Per-FedAvg Moderate non-IID (α=0.5)            0.0      0.1318      50
  SCAFFOLD  Extreme non-IID (α=0.1)            0.0      0.1603      50
  SCAFFOLD              IID (α=1.0)            0.0      0.2196      50
  SCAFFOLD Moderate non-IID (α=0.5)            0.0      0.1238      50

Table saved to: /content/drive/

## Stage 3 — Figure 5.1: Convergence Comparison Chart

In [ ]:
algorithms = ['fedavg','fedprox','scaffold','perfedavg']
conditions = ['iid','mod_noniid','ext_noniid']
styles     = ['-','--',':','-.']
colors     = ['#111111','#444444','#777777','#AAAAAA']

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax, cond in zip(axes, conditions):
    for i, alg in enumerate(algorithms):
        key = f"{alg}_{cond}"
        if key in all_results and all_results[key].get('losses'):
            rounds = [r for r,_ in all_results[key]['losses']]
            losses = [l for _,l in all_results[key]['losses']]
            ax.plot(rounds, losses, linestyle=styles[i], color=colors[i],
                    linewidth=1.8, label=alg_labels.get(alg,alg))
    ax.set_title(cond_labels.get(cond,cond), fontsize=10, fontweight='bold')
    ax.set_xlabel('Communication round')
    ax.grid(axis='y')
    if ax == axes[0]:
        ax.set_ylabel('Aggregated loss')
        ax.legend(loc='upper right', fontsize=8)

fig.suptitle('Figure 5.1: FL Algorithm Convergence Across Data Heterogeneity Conditions',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR + 'Figure_5_1_Convergence.png')
plt.close()
print("Figure 5.1 saved")

Figure 5.1 saved


## Stage 4 — Figure 5.2: Algorithm Performance Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

alg_order  = ['FedAvg','FedProx','SCAFFOLD','Per-FedAvg']
cond_order = ['IID (α=1.0)','Moderate non-IID (α=0.5)','Extreme non-IID (α=0.1)']
fills      = ['#FFFFFF','#AAAAAA','#555555']
hatches    = ['','///','...']
x = np.arange(len(alg_order))
width = 0.25

for j, cond in enumerate(cond_order):
    vals = []
    for alg in alg_order:
        match = df_results[(df_results['Algorithm']==alg) &
                           (df_results['Condition']==cond)]
        val = match['Final AUC-ROC'].values[0] if len(match)>0 else 0
        vals.append(float(val) if val != 'N/A' else 0)
    ax.bar(x + (j-1)*width, vals, width*0.9,
           color=fills[j], edgecolor='#111111', linewidth=0.6,
           hatch=hatches[j], label=cond, zorder=2)

ax.set_xticks(x)
ax.set_xticklabels(alg_order)
ax.set_ylabel('AUC-ROC (held-out test set)')
ax.set_ylim(0, 1.05)
ax.axhline(0.80, color='#555', linewidth=0.7, linestyle='--', alpha=0.7)
ax.text(3.4, 0.81, 'Clinical threshold (0.80)', fontsize=8, color='#555')
ax.legend(frameon=True, fontsize=9)
ax.grid(axis='y', zorder=0)
ax.set_title('Figure 5.2: FL Algorithm AUC-ROC Performance by Data Heterogeneity Condition')
plt.tight_layout()
plt.savefig(OUTPUTS_DIR + 'Figure_5_2_Algorithm_Comparison.png')
plt.close()
print("Figure 5.2 saved")

Figure 5.2 saved


## Stage 5 — Figure 5.3: Privacy-Utility Trade-off

In [ ]:
# Load DP results
try:
    with open(RESULTS_DIR + 'dp_results.json') as f:
        dp_data = json.load(f)

    targets = [r['target_epsilon'] for r in dp_data]
    actuals = [r['actual_epsilon'] for r in dp_data]
    aucs    = [r['auc_roc']        for r in dp_data]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(targets, aucs, color='#111111', marker='o', linewidth=2,
            markersize=7, markerfacecolor='white', markeredgewidth=1.5,
            label='DP-FL (experimental)', zorder=3)

    ax.axhline(0.80, color='#555', linewidth=0.8, linestyle='--', alpha=0.7)
    ax.text(max(targets)*0.5, 0.805, 'Clinical threshold (AUC ≥ 0.80)',
            fontsize=8, color='#555')

    for te, ae, a in zip(targets, actuals, aucs):
        ax.annotate(f'ε={ae:.2f}', (te, a),
                    textcoords='offset points', xytext=(0,10),
                    ha='center', fontsize=8, color='#333')

    ax.set_xlabel('Target ε (privacy budget) — lower = stronger privacy')
    ax.set_ylabel('AUC-ROC on held-out test set')
    ax.set_title('Figure 5.3: Privacy-Utility Trade-off\nDifferential Privacy ε vs AUC-ROC (moderate non-IID conditions)')
    ax.grid(axis='y')
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR + 'Figure_5_3_DP_Tradeoff.png')
    plt.close()
    print("Figure 5.3 saved")
except FileNotFoundError:
    print("dp_results.json not found — run Notebook 03 first")

Figure 5.3 saved


## Stage 6 — Figure 5.4: MIA and HE Summary

In [ ]:
# MIA summary
try:
    with open(RESULTS_DIR + 'mia_results.json') as f:
        mia = json.load(f)
    print("MEMBERSHIP INFERENCE ATTACK SUMMARY")
    print("="*50)
    print(f"Attack AUC-ROC:      {mia['attack_auc']:.4f}")
    print(f"Adversary advantage: {mia['advantage']:.4f}")
    print(f"Risk level:          {mia['risk_level']}")
    print(f"(Baseline random chance = 0.5000)")
except FileNotFoundError:
    print("mia_results.json not found — run Notebook 03 first")

# HE overhead summary
try:
    with open(RESULTS_DIR + 'he_results.json') as f:
        he = json.load(f)
    print("\nHOMOMORPHIC ENCRYPTION OVERHEAD")
    print("="*50)
    for k, v in he.items():
        print(f"{k}: enc={v['enc_seconds']}s  dec={v['dec_seconds']}s  "
              f"error={v['max_error']:.2e}")
except FileNotFoundError:
    print("he_results.json not found — run Notebook 03 first")

MEMBERSHIP INFERENCE ATTACK SUMMARY
Attack AUC-ROC:      0.3983
Adversary advantage: 0.2035
Risk level:          LOW — acceptable for NHS deployment
(Baseline random chance = 0.5000)

HOMOMORPHIC ENCRYPTION OVERHEAD
chunk_1000: enc=0.007s  dec=0.0018s  error=6.78e-09
chunk_5000: enc=0.0141s  dec=0.0034s  error=7.80e-09
chunk_11970: enc=0.0211s  dec=0.0055s  error=7.92e-09


## Stage 7 — Final Output Summary

In [ ]:
print("ALL OUTPUTS SAVED TO GOOGLE DRIVE")
print("="*55)
for f in sorted(os.listdir(OUTPUTS_DIR)):
    size = os.path.getsize(OUTPUTS_DIR + f)
    print(f"  {f:<45} {size/1024:>6.1f} KB")

print(f"\nLocation: {OUTPUTS_DIR}")
print("\nNotebook 04 COMPLETE")
print("Your dissertation results are ready for Chapter 5 (Findings)")

ALL OUTPUTS SAVED TO GOOGLE DRIVE
  Figure_5_1_Convergence.png                     356.8 KB
  Figure_5_2_Algorithm_Comparison.png             93.4 KB
  Figure_5_3_DP_Tradeoff.png                     129.9 KB
  Table_5_1_Results_Summary.csv                    0.6 KB

Location: /content/drive/MyDrive/FL_Dissertation/outputs/

Notebook 04 COMPLETE
Your dissertation results are ready for Chapter 5 (Findings)


In [ ]:
import json, os
import numpy as np

RESULTS_DIR = '/content/drive/MyDrive/FL_Dissertation/results/'
ARTEFACTS   = '/content/drive/MyDrive/FL_Dissertation/data/artefacts/'
CLEAN_DIR   = '/content/drive/MyDrive/FL_Dissertation/data/cleaned/'

print("="*65)
print("CORRECTED RESULTS SUMMARY")
print("="*65)

# Dataset stats
X_train = np.load(CLEAN_DIR + 'X_train.npy')
y_train = np.load(CLEAN_DIR + 'y_train.npy')
X_test  = np.load(CLEAN_DIR + 'X_test.npy')
y_test  = np.load(CLEAN_DIR + 'y_test.npy')
print(f"\nDATASET (post leakage fix):")
print(f"  Training:      {X_train.shape[0]:,} x {X_train.shape[1]} features")
print(f"  Test:          {X_test.shape[0]:,} x {X_test.shape[1]} features")
print(f"  Positive rate: {y_train.mean():.4f}")

# Metadata
with open(ARTEFACTS + 'metadata.json') as f:
    meta = json.load(f)
print(f"  Input dim:     {meta['input_dim']}")

# Feature importance
import pandas as pd
fi = pd.read_csv(ARTEFACTS + 'feature_importance.csv')
print(f"\nTOP 10 FEATURES (after discharge_location removed):")
print(fi.head(10).to_string(index=False))

# FL results
print(f"\nFL SIMULATION RESULTS:")
files = ['fedavg','fedprox','scaffold','perfedavg']
conds = ['iid','mod_noniid','ext_noniid']
for alg in files:
    for cond in conds:
        fname = f"{alg}_{cond}.json"
        path  = RESULTS_DIR + fname
        if os.path.exists(path):
            with open(path) as f:
                d = json.load(f)
            losses = d.get('losses', [])
            aucs   = d.get('auc',    [])
            fl     = losses[-1][1] if losses else 0
            fa     = aucs[-1][1]   if aucs   else 0
            ba     = max(v for _,v in aucs) if aucs else 0
            t80    = next((r for r,v in aucs if v>=0.80),'not reached')
            print(f"  {alg}_{cond:<18} loss={fl:.4f}  "
                  f"final_auc={fa:.4f}  best_auc={ba:.4f}  "
                  f"rounds_to_0.80={t80}")
        else:
            print(f"  {alg}_{cond:<18} NOT FOUND")

# Privacy
print(f"\nDP RESULTS:")
with open(RESULTS_DIR + 'dp_results.json') as f:
    dp = json.load(f)
for r in dp:
    print(f"  eps={r['target_epsilon']}  actual={r.get('actual_epsilon',0):.4f}"
          f"  AUC={r.get('auc_roc',0):.4f}")

print(f"\nMIA RESULT:")
with open(RESULTS_DIR + 'mia_results.json') as f:
    mia = json.load(f)
print(f"  Attack AUC={mia['attack_auc']:.4f}  "
      f"Advantage={mia['advantage']:.4f}  "
      f"Risk={mia.get('risk_level','N/A')}")

print(f"\nTEST SET EVALUATION (centralised baseline):")
with open(RESULTS_DIR + 'test_evaluation.json') as f:
    te = json.load(f)
for k,v in te.items():
    print(f"  {k}: {v:.4f}")

print("\n" + "="*65)
print("END — paste this entire output into the chat")

CORRECTED RESULTS SUMMARY

DATASET (post leakage fix):
  Training:      436,822 x 26 features
  Test:          109,206 x 26 features
  Positive rate: 0.0216
  Input dim:     26

TOP 10 FEATURES (after discharge_location removed):
                feature  mi_score
          gender_female  0.088519
               language  0.075710
     insurance_Medicare  0.073904
 marital_status_MARRIED  0.067623
                   race  0.057551
  marital_status_SINGLE  0.051634
     admission_location  0.038018
admission_type_EW EMER.  0.037982
      insurance_Private  0.037321
        age_group_41-60  0.036126

FL SIMULATION RESULTS:
  fedavg_iid                loss=0.2248  final_auc=0.0000  best_auc=0.0000  rounds_to_0.80=not reached
  fedavg_mod_noniid         loss=0.1243  final_auc=0.0000  best_auc=0.0000  rounds_to_0.80=not reached
  fedavg_ext_noniid         loss=0.1591  final_auc=0.0000  best_auc=0.0000  rounds_to_0.80=not reached
  fedprox_iid                loss=0.1617  final_auc=0.0000  bes

for pix

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

# ── Paths ─────────────────────────────────────────────────────────────────
CLEAN_DIR  = '/content/drive/MyDrive/FL_Dissertation/data/cleaned/'
ARTEFACTS  = '/content/drive/MyDrive/FL_Dissertation/data/artefacts/'
OUTPUT_DIR = '/content/drive/MyDrive/FL_Dissertation/outputs/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Load actual data ────────────────────────────────────────────────────────
y_train = np.load(CLEAN_DIR + 'y_train.npy')
y_test  = np.load(CLEAN_DIR + 'y_test.npy')
y_all   = np.concatenate([y_train, y_test])

fi = pd.read_csv(ARTEFACTS + 'feature_importance.csv')
top10 = fi.head(10)

# ── Figure style ────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.dpi': 200,
    'savefig.bbox': 'tight'
})

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── LEFT PANEL: Class distribution ─────────────────────────────────────────
survived = (y_all == 0).sum()
died     = (y_all == 1).sum()
total    = len(y_all)

bars = axes[0].bar(
    ['Survived (0)', 'Died (1)'],
    [survived, died],
    color=['#2E75B6', '#C00000'],
    width=0.5,
    edgecolor='white'
)

# Add count and percentage labels on bars
for bar, count in zip(bars, [survived, died]):
    pct = count / total * 100
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 2000,
        f'{count:,}\n({pct:.1f}%)',
        ha='center', va='bottom',
        fontsize=12, fontweight='bold',
        color='black'
    )

axes[0].set_title('Class Distribution — In-Hospital Mortality',
                   fontweight='bold', fontsize=13, pad=15)
axes[0].set_ylabel('Number of Admissions', fontsize=11)
axes[0].set_ylim(0, survived * 1.15)
axes[0].yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f'{int(x):,}')
)
axes[0].grid(axis='y', alpha=0.3, linestyle='--')

# Annotation
axes[0].text(
    0.97, 0.97,
    f'Total: {total:,}\nPositive rate: {died/total*100:.2f}%',
    transform=axes[0].transAxes,
    ha='right', va='top', fontsize=10,
    color='#595959', style='italic',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='#EBF3FB',
              edgecolor='#BDD7EE', alpha=0.8)
)

# ── RIGHT PANEL: Top 10 features by MI score ───────────────────────────────
features = top10['feature'].tolist()
scores   = top10['mi_score'].tolist()

# Reverse so highest is at top
features_rev = features[::-1]
scores_rev   = scores[::-1]

# Colour bars by score magnitude
norm_scores = [(s - min(scores)) / (max(scores) - min(scores) + 1e-9)
               for s in scores_rev]
colours = [plt.cm.Blues(0.4 + 0.5 * ns) for ns in norm_scores]

hbars = axes[1].barh(
    range(len(features_rev)),
    scores_rev,
    color=colours,
    edgecolor='white',
    height=0.65
)

# Add value labels
for i, (bar, score) in enumerate(zip(hbars, scores_rev)):
    axes[1].text(
        score + 0.001,
        bar.get_y() + bar.get_height() / 2,
        f'{score:.4f}',
        va='center', ha='left',
        fontsize=9.5, color='#333333'
    )

axes[1].set_yticks(range(len(features_rev)))
axes[1].set_yticklabels(features_rev, fontsize=10)
axes[1].set_xlabel('Mutual Information Score', fontsize=11)
axes[1].set_title('Top 10 Predictive Features — Mutual Information Score',
                   fontweight='bold', fontsize=13, pad=15)
axes[1].grid(axis='x', alpha=0.3, linestyle='--')
axes[1].set_xlim(0, max(scores_rev) * 1.18)

# Highlight top feature
axes[1].get_yticklabels()[-1].set_color('#C00000')
axes[1].get_yticklabels()[-1].set_fontweight('bold')

# ── Shared footer ──────────────────────────────────────────────────────────
fig.suptitle(
    'Figure 4.1: Class Distribution and Top 10 Feature Importance Scores\n'
    'Source: MIMIC-IV v3.1 — post-preprocessing, leakage-corrected dataset',
    fontsize=11, y=-0.02, color='#595959', style='italic'
)

plt.tight_layout(rect=[0, 0.02, 1, 1])

# ── Save ───────────────────────────────────────────────────────────────────
out_path = OUTPUT_DIR + 'Figure_4_1_Class_Distribution_Feature_Importance.png'
plt.savefig(out_path, dpi=200, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()

print(f"\nSaved: {out_path}")
print(f"File size: {os.path.getsize(out_path)/1024:.1f} KB")


Saved: /content/drive/MyDrive/FL_Dissertation/outputs/Figure_4_1_Class_Distribution_Feature_Importance.png
File size: 252.9 KB
